# Demo interactiva: predicción de ratings en nuevas reseñas

Esta demo convierte el modelo entrenado en una pequeña aplicación interactiva. El usuario escribe una reseña de restaurante y el notebook devuelve el rating estimado, el texto normalizado que recibe el modelo y la distribución de probabilidades entre las cinco clases.

## Objetivo de la demostración

La finalidad es observar cómo se comporta el clasificador ante ejemplos nuevos y conectar los resultados de la evaluación con un caso de uso sencillo. La demo no vuelve a entrenar el modelo: utiliza el archivo guardado en `data/mejor_modelo.joblib`.


In [1]:
import re
import html
import joblib
import numpy as np
import matplotlib.pyplot as plt
import joblib
import ipywidgets as widgets
from IPython.display import display, clear_output

## 1. Preparación de una predicción

Antes de predecir, la reseña nueva debe pasar por la misma normalización utilizada durante el entrenamiento. La función elimina ruido como URLs y correos electrónicos, expande contracciones, convierte el texto a minúsculas, sustituye algunos emojis por etiquetas y limita los alargamientos de caracteres.

A continuación, el modelo recibe una lista con la reseña normalizada y devuelve:

- **Rating predicho:** la clase con mayor probabilidad.
- **Probabilidades por clase:** distribución estimada para 1, 2, 3, 4 y 5 estrellas.
- **Texto normalizado:** versión que permite comprobar qué información llegó al vectorizador.

In [2]:
EMOJI_MAP = {
    "😊": " emo_pos ", "😃": " emo_pos ", "😍": " emo_pos ", "👍": " emo_pos ",
    "❤️": " emo_pos ", "🔥": " emo_pos ", "😋": " emo_pos ", "🙌": " emo_pos ",
    "😞": " emo_neg ", "😡": " emo_neg ", "👎": " emo_neg ", "😢": " emo_neg ",
    "🤢": " emo_neg ", "😠": " emo_neg ",
}
CONTRACTIONS = {
    "won't": "will not", "can't": "cannot", "n't": " not",
    "'re": " are", "'s": " is", "'d": " would", "'ll": " will",
    "'t": " not", "'ve": " have", "'m": " am",
}
 
def normalizar(texto: str) -> str:
    if not isinstance(texto, str):
        return ""
    t = html.unescape(texto)
    t = re.sub(r"http\S+|www\.\S+", " ", t)
    t = re.sub(r"\S+@\S+", " ", t)
    for emo, tag in EMOJI_MAP.items():
        t = t.replace(emo, tag)
    t = t.lower()
    for k, v in CONTRACTIONS.items():
        t = t.replace(k, v)
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)
    t = re.sub(r"[^a-záéíóúñü\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def predecir_reseña(texto_nuevo: str, modelo):
    """Limpia el texto igual que en entrenamiento y devuelve:
    (rating predicho, probabilidades por clase, texto ya limpio)."""
    limpio = normalizar(texto_nuevo)
    proba = modelo.predict_proba([limpio])[0]
    pred = modelo.predict([limpio])[0]
    return pred, proba, limpio

In [3]:
MODELO_DEMO = joblib.load("data/mejor_modelo.joblib")

## 2. Widget interactivo

El widget proporciona una interfaz mínima para probar el clasificador:

1. Escribe una reseña en inglés en el cuadro de texto.
2. Pulsa **Predecir rating**.
3. Revisa el texto original, el texto normalizado y la distribución de probabilidades.

La clase predicha aparece resaltada en verde. Las barras restantes permiten detectar casos ambiguos: si varias clases tienen probabilidades parecidas, el modelo está tomando una decisión con poca separación entre alternativas.

In [ ]:
caja_texto = widgets.Textarea(
    value="",
    placeholder="Escribe aquí una reseña de restaurante en inglés...",
    description="Reseña:",
    layout=widgets.Layout(width="600px", height="100px"),
)
 
boton_predecir = widgets.Button(
    description="Predecir rating",
    button_style="success",
    icon="check",
)
 
salida = widgets.Output()
 
 
def al_presionar_boton(b):
    with salida:
        clear_output(wait=True)   # borra el resultado anterior antes de mostrar el nuevo
 
        texto = caja_texto.value.strip()
        if not texto:
            print("Escribe una reseña antes de predecir.")
            return
 
        pred, proba, limpio = predecir_reseña(texto, MODELO_DEMO)
 
        print(f"Texto original: {texto}")
        print(f"Texto normalizado (lo que ve el modelo): {limpio}")
        print(f"\n⭐ Predicción del modelo: {pred} estrella(s)")
 
        # Gráfico de barras con la confianza del modelo en cada clase posible.
        # Esto es más informativo que solo mostrar la clase ganadora: si el
        # modelo predijo 4★ pero con 35% vs. 30% en 5★, es una decisión
        # insegura, no una certeza — y eso se ve de inmediato en la barra.
        fig, ax = plt.subplots(figsize=(6, 3))
        colores = ["#c44e52" if c != pred else "#55a868" for c in MODELO_DEMO.classes_]
        ax.bar(MODELO_DEMO.classes_.astype(str), proba, color=colores, edgecolor="white")
        ax.set_xlabel("estrellas"); ax.set_ylabel("probabilidad")
        ax.set_ylim(0, 1)
        ax.set_title("Confianza del modelo por clase")
        for i, p in enumerate(proba):
            ax.text(i, p + 0.02, f"{p:.0%}", ha="center", fontsize=9)
        plt.tight_layout(); plt.show()
 
 
boton_predecir.on_click(al_presionar_boton)
 
print("Demo interactiva del clasificador de reseñas")
print(f"Modelo en uso: {MODELO_DEMO.named_steps['clf'].__class__.__name__} "
      f"sobre {MODELO_DEMO.named_steps['tfidf'].__class__.__name__}\n")
display(caja_texto, boton_predecir, salida)

Demo interactiva del clasificador de reseñas
Modelo en uso: LogisticRegression sobre TfidfVectorizer



Textarea(value='', description='Reseña:', layout=Layout(height='100px', width='600px'), placeholder='Escribe a…

Button(button_style='success', description='Predecir rating', icon='check', style=ButtonStyle())

Output()

## 3. Cómo leer la salida

La salida muestra tres niveles de información:

- El **texto original** permite comprobar exactamente qué se introdujo.
- El **texto normalizado** permite verificar la transformación previa a la predicción.
- El **gráfico de probabilidades** muestra la competencia entre las cinco estrellas.

La barra verde identifica la clase elegida, pero no debe leerse de forma aislada. Una probabilidad claramente dominante indica una decisión más definida; varias barras similares indican incertidumbre y aconsejan revisar la reseña manualmente.

Además, `predict_proba` representa la distribución interna estimada por la regresión logística.

## 4. Casos de prueba

Estos ejemplos originales permiten probar manualmente la demo con distintos tipos de lenguaje:

### Muy positiva

> Absolutely amazing experience! The food was incredible and the staff went above and beyond. Will definitely come back!

### Muy negativa

> Terrible service, waited over an hour and the food was cold when it finally arrived. Never coming back.

### Mixta: comida buena, servicio malo

> The food was actually pretty good, but the service was so slow and the waiter was rude. Not worth the wait.

### Neutra o ambigua

> It was okay. Nothing special, but nothing terrible either. Prices are fair for what you get.
